# IHD compartment refinement and intermediate annotation

This notebook reproduces the manual compartment-specific refinement stage from the IHD round-2 annotation. It reads `adata_IHD_annotated.h5ad`, applies the recorded fibroblast, cardiomyocyte, vascular and immune refinements, and saves `adata_celltype_corrected_v3.h5ad` with `celltype_final_corrected_v3`.

Run after `01_Xenium_cell_type_annotation_DCM_IHD.ipynb`. The cluster-to-cell-type mappings below are retained from the supplied source; they are not inferred or newly invented.

## Load the first-pass IHD object

In [ ]:
import scanpy as sc

adata = sc.read_h5ad("adata_IHD_annotated.h5ad")


## Fibroblast compartment refinement

In [ ]:
# Initialize the refinement column from the first-pass broad annotation.
# Subsequent manual refinement steps update this column.
adata.obs["celltype_final"] = adata.obs["celltype"].astype(str)

fb = adata[adata.obs["celltype_final"] == "FB"].copy()

sc.pp.normalize_total(fb, target_sum=1e4)
sc.pp.log1p(fb)

sc.tl.pca(fb, n_comps=30)

sc.pp.neighbors(fb, n_neighbors=15, n_pcs=30)
sc.tl.leiden(fb, resolution=2.0, flavor="leidenalg")

sc.pl.umap(fb, color=["leiden", "NPPA", "COL1A1", "DCN"])


In [ ]:
# Keep only genes that are present in the dataset
def filter_genes(gene_list, adata):
    return [g for g in gene_list if g in adata.var_names]

cm_markers = filter_genes(
    ["NPPA", "NPPB", "TNNT2", "MYH6", "MYH7"], fb
)

fb_markers = filter_genes(
    ["COL1A1", "COL1A2", "DCN", "LUM", "VCAN"], fb
)

stress_markers = filter_genes(
    ["PECAM1", "VWF", "CD93"], fb
)

print("CM:", cm_markers)
print("FB:", fb_markers)
print("Stress:", stress_markers)

import scanpy as sc

markers = {
    "CM_markers": cm_markers,
    "FB_markers": fb_markers,
    "Stress_markers": stress_markers
}

# Flatten the list for Scanpy
marker_list = sum(markers.values(), [])

sc.pl.dotplot(
    fb,
    marker_list,
    groupby="leiden",
    standard_scale="var",   # extrem wichtig!
    dot_min=0.05,
    dot_max=0.8,
    cmap="Reds"
)


In [ ]:
sc.tl.rank_genes_groups(
    fb,
    groupby="leiden",
    method="wilcoxon",
    n_genes=20
)

sc.pl.rank_genes_groups(
    fb,
    sharey=False,
    title="Ranked genes"
)


In [ ]:
import scanpy as sc

# --- Marker sets ---
fb_genes = ["COL1A1", "COL1A2", "DCN", "LUM", "PDGFRA"]

cm_genes = ["TNNT2", "MYH6", "MYH7", "NPPA", "NPPB"]

vascular_genes = ["PECAM1", "VWF", "CD93"]

immune_genes = ["PTPRC", "LYZ", "CD3D", "CD3E", "NKG7"]

# --- keep only existing genes ---
def present(g, adata):
    return [x for x in g if x in adata.var_names]

fb_p = present(fb_genes, adata)
cm_p = present(cm_genes, adata)
vasc_p = present(vascular_genes, adata)
imm_p = present(immune_genes, adata)

print("FB:", fb_p)
print("CM:", cm_p)
print("Vascular:", vasc_p)
print("Immune:", imm_p)

# --- scores ---
sc.tl.score_genes(adata, fb_p, score_name="FB_score")
sc.tl.score_genes(adata, cm_p, score_name="CM_score")
sc.tl.score_genes(adata, vasc_p, score_name="Vascular_score")
sc.tl.score_genes(adata, imm_p, score_name="Immune_score")

# --- derived identities (sehr wichtig) ---
adata.obs["CM_minus_FB"] = adata.obs["CM_score"] - adata.obs["FB_score"]
adata.obs["Vasc_minus_FB"] = adata.obs["Vascular_score"] - adata.obs["FB_score"]
adata.obs["Immune_minus_FB"] = adata.obs["Immune_score"] - adata.obs["FB_score"]

# --- transfer to fb ---
for col in ["FB_score","CM_score","Vascular_score","Immune_score",
            "CM_minus_FB","Vasc_minus_FB","Immune_minus_FB"]:
    fb.obs[col] = adata.obs.loc[fb.obs_names, col]

# --- cluster summary (26 leiden clusters) ---
score_table = (
    fb.obs
    .groupby("leiden")[[
        "CM_score","FB_score","Vascular_score","Immune_score",
        "CM_minus_FB","Vasc_minus_FB","Immune_minus_FB"
    ]]
    .mean()
    .sort_values("CM_score", ascending=False)
)

print(score_table.round(3))


In [ ]:
import numpy as np
import pandas as pd

# --- 1. Cluster → Celltype Mapping (FB-subset result) ---
cluster_to_celltype = {
    "4": "CM",
    "16": "CM",
    "21": "CM",
    "15": "vascular"
}

# --- 2. FB reannotation ---
fb.obs["celltype_new"] = (
    fb.obs["leiden"]
    .astype(str)
    .map(cluster_to_celltype)
    .fillna("FB")
)

print("FB reannotation:")
print(fb.obs["celltype_new"].value_counts())

# --- 3. OVERWRITE ORIGINAL CELLTYPE in adata ---
adata.obs.loc[fb.obs_names, "celltype"] = fb.obs["celltype_new"].values

# --- 4. (optional cleanup: delete helper column if you want clean object) ---
# fb.obs.drop(columns=["celltype_new"], inplace=True)

# --- 5. global distribution ---
print("\nGlobal distribution (UPDATED celltype):")
print(adata.obs["celltype"].value_counts())

# --- 6. fb sanity check ---
fb_check = (
    fb.obs.groupby("leiden")["celltype_new"]
    .value_counts()
    .unstack(fill_value=0)
)

print("\nFB cluster composition:")
print(fb_check)

# --- 7. cross-check ---
print("\nCross-check:")
print(pd.crosstab(adata.obs["celltype"], adata.obs["celltype"]))


# The first-pass annotation notebook stores the broad labels in `celltype`.
# The original compartment-refinement code uses `celltype_final` as its source
# label, so carry the updated fibroblast reannotation into that column.
adata.obs["celltype_final"] = adata.obs["celltype"].astype(str)


In [ ]:
print(len(fb.obs_names))
print(len(set(fb.obs_names) & set(adata.obs_names)))
print(fb.obs_names.isin(adata.obs_names).mean())


In [ ]:
# --- 5. Speichern ---
adata.write("adata_global_fb_cleaned_annotated.h5ad")


In [ ]:
adata = sc.read_h5ad("adata_global_fb_cleaned_annotated.h5ad")

if "celltype_final" not in adata.obs.columns:
    raise KeyError("Required column 'celltype_final' is missing before cardiomyocyte refinement.")


## Immune marker review

In [ ]:
import pandas as pd

# --- immune gene panel (extended, but safe) ---
immune_panel = [
    # pan-immune
    "PTPRC",

    # T cells
    "CD3D", "CD3E", "CD4", "CD8A", "CD8B",

    # NK / cytotoxic
    "NKG7", "GNLY", "PRF1", "GZMB",

    # myeloid / macrophage / monocyte
    "LYZ", "LST1", "S100A8", "S100A9", "CD68", "CSF1R", "FCGR3A",

    # dendritic cells
    "HLA-DRA", "HLA-DPB1", "ITGAX"
]

# --- check presence in dataset ---
present_genes = [g for g in immune_panel if g in adata.var_names]
missing_genes = [g for g in immune_panel if g not in adata.var_names]

print("\n=== IMMUNE MARKER PRESENCE CHECK ===")
print(f"Total immune markers defined: {len(immune_panel)}")
print(f"Present in dataset: {len(present_genes)}")
print(f"Missing: {len(missing_genes)}\n")

print("✔ Present genes:")
print(present_genes)

print("\n✖ Missing genes:")
print(missing_genes)


In [ ]:
adata
print("CM in adata:", (adata.obs["celltype"] == "CM").sum())


## Cardiomyocyte compartment refinement

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

def present(genes, adata):
    return [g for g in genes if g in adata.var_names]

# --- subset ---
cm_sub = adata[adata.obs["celltype_final"] == "CM"].copy()
cm_sub


In [ ]:
# --- preprocessing (Xenium-friendly) ---
sc.pp.normalize_total(cm_sub, target_sum=1e4)
sc.pp.log1p(cm_sub)

# --- clustering (NO HVG, NO scale) ---
sc.tl.pca(cm_sub, n_comps=30)
sc.pp.neighbors(cm_sub, n_neighbors=15, n_pcs=30)
sc.tl.leiden(cm_sub, resolution=2.0, flavor="leidenalg")
sc.tl.umap(cm_sub)

# --- marker sets ---
cm_markers = present(["TNNT2","MYH6","MYH7","NPPA","NPPB"], cm_sub)
fb_markers = present(["COL1A1","DCN","LUM","PDGFRA"], cm_sub)
vasc_markers = present(["PECAM1","VWF","CD93"], cm_sub)
imm_markers = present(["PTPRC","CD3D","CD3E","NKG7"], cm_sub)

print("CM:", cm_markers)
print("FB:", fb_markers)
print("Vascular:", vasc_markers)
print("Immune:", imm_markers)

# --- robust expression-based scoring (recommended for Xenium) ---
cm_sub.obs["CM_mean"] = np.array(cm_sub[:, cm_markers].X.mean(axis=1)).flatten()
cm_sub.obs["FB_mean"] = np.array(cm_sub[:, fb_markers].X.mean(axis=1)).flatten()
cm_sub.obs["Vasc_mean"] = np.array(cm_sub[:, vasc_markers].X.mean(axis=1)).flatten()
cm_sub.obs["Immune_mean"] = np.array(cm_sub[:, imm_markers].X.mean(axis=1)).flatten()

# --- optional scanpy scores (secondary, not primary) ---
sc.tl.score_genes(cm_sub, cm_markers, score_name="CM_score")
sc.tl.score_genes(cm_sub, fb_markers, score_name="FB_score")
sc.tl.score_genes(cm_sub, vasc_markers, score_name="Vascular_score")
sc.tl.score_genes(cm_sub, imm_markers, score_name="Immune_score")

# --- derived metrics (use mean-based = more stable) ---
cm_sub.obs["CM_minus_FB"] = cm_sub.obs["CM_mean"] - cm_sub.obs["FB_mean"]
cm_sub.obs["CM_minus_Vasc"] = cm_sub.obs["CM_mean"] - cm_sub.obs["Vasc_mean"]
cm_sub.obs["Vasc_minus_FB"] = cm_sub.obs["Vasc_mean"] - cm_sub.obs["FB_mean"]
cm_sub.obs["Immune_minus_FB"] = cm_sub.obs["Immune_mean"] - cm_sub.obs["FB_mean"]

# --- cluster summary ---
cm_scores = (
    cm_sub.obs
    .groupby("leiden")[[
        "CM_mean","FB_mean","Vasc_mean","Immune_mean",
        "CM_minus_FB","CM_minus_Vasc","Vasc_minus_FB"
    ]]
    .mean()
    .sort_values("CM_minus_FB", ascending=False)
)

print(cm_scores.round(3))

# --- visualization ---
sc.pl.umap(
    cm_sub,
    color=[
        "leiden",
        "CM_mean",
        "FB_mean",
        "Vasc_mean",
        "CM_minus_FB"
    ],
    vmax="p99"
)


In [ ]:
import scanpy as sc

# =========================
# Marker sets (CM context)
# =========================
markers = {
    "CM": ["TNNT2", "MYH6", "MYH7", "NPPA", "NPPB"],
    "FB": ["COL1A1", "DCN", "LUM", "PDGFRA"],
    "Vascular": ["PECAM1", "VWF", "CD93"],
    "Immune": ["PTPRC", "CD3D", "CD8A", "NKG7", "CD68", "FCGR3A"]
}

marker_list = sum(markers.values(), [])

# =========================
# 1. Marker Dotplot
# =========================
sc.pl.dotplot(
    cm_sub,
    marker_list,
    groupby="leiden",
    standard_scale="var",
    dot_min=0.05,
    dot_max=0.8,
    cmap="Reds",
    title="CM subset: marker genes (CM / FB / vascular / immune)"
)

# =========================
# 2. Rank genes
# =========================
sc.tl.rank_genes_groups(
    cm_sub,
    groupby="leiden",
    method="wilcoxon",
    n_genes=10
)

# =========================
# 3. Ranked genes dotplot
# =========================
sc.pl.rank_genes_groups_dotplot(
    cm_sub,
    n_genes=5,
    standard_scale="var",
    cmap="Reds",
    title="CM subset: top ranked genes per cluster"
)

# =========================
# 4. Optional combined view (markers + ranked genes)
# =========================
ranked = cm_sub.uns["rank_genes_groups"]["names"]
top_ranked = list(set(ranked.flatten()))[:30]

combined_genes = marker_list + top_ranked

sc.pl.dotplot(
    cm_sub,
    combined_genes,
    groupby="leiden",
    standard_scale="var",
    dot_min=0.05,
    dot_max=0.8,
    cmap="Reds",
    title="CM subset: markers + ranked genes"
)


In [ ]:
sc.tl.rank_genes_groups(
    cm_sub,
    groupby="leiden",
    method="wilcoxon",
    n_genes=20
)

sc.pl.rank_genes_groups(
    cm_sub,
    sharey=False,
    title="Ranked genes"
)


In [ ]:
adata_c9 = cm_sub[cm_sub.obs["leiden"] == "12"].copy()


In [ ]:
sc.pl.umap(adata_c9, color=["TNNT2", "PECAM1", "VWF"], vmax="p99")


In [ ]:
import matplotlib.pyplot as plt

plt.scatter(
    adata_c9[:, "TNNT2"].X.toarray(),
    adata_c9[:, "PECAM1"].X.toarray(),
    s=5, alpha=0.5
)
plt.xlabel("TNNT2")
plt.ylabel("PECAM1")
plt.show()


In [ ]:
sc.pp.scale(adata_c9)
sc.tl.pca(adata_c9)
sc.pp.neighbors(adata_c9)
sc.tl.leiden(adata_c9, resolution=0.5, flavor="leidenalg")

sc.pl.umap(adata_c9, color=["leiden", "TNNT2", "PECAM1"])


In [ ]:
sc.tl.rank_genes_groups(adata_c9, "leiden", method="wilcoxon")
sc.pl.rank_genes_groups(adata_c9, n_genes=10)


In [ ]:
adata_c11 = cm_sub[cm_sub.obs["leiden"] == "11"].copy()


In [ ]:
sc.pp.scale(adata_c11)
sc.tl.pca(adata_c11)
sc.pp.neighbors(adata_c11)
sc.tl.leiden(adata_c11, resolution=1, flavor="leidenalg")

sc.pl.umap(adata_c11, color=["leiden", "TNNT2", "PECAM1"])


In [ ]:
sc.tl.rank_genes_groups(adata_c11, "leiden", method="wilcoxon")
sc.pl.rank_genes_groups(adata_c11, n_genes=10)


In [ ]:
adata_c12 = cm_sub[cm_sub.obs["leiden"] == "12"].copy()


In [ ]:
sc.pp.scale(adata_c12)
sc.tl.pca(adata_c12)
sc.pp.neighbors(adata_c12)
sc.tl.leiden(adata_c12, resolution=1, flavor="leidenalg")

sc.pl.umap(adata_c12, color=["leiden", "TNNT2", "PECAM1"])


In [ ]:
sc.tl.rank_genes_groups(adata_c12, "leiden", method="wilcoxon")
sc.pl.rank_genes_groups(adata_c12, n_genes=10)


In [ ]:
adata_c17 = cm_sub[cm_sub.obs["leiden"] == "14"].copy()


In [ ]:
import matplotlib.pyplot as plt

plt.scatter(
    adata_c17[:, "TNNT2"].X.toarray(),
    adata_c17[:, "PECAM1"].X.toarray(),
    s=5, alpha=0.5
)
plt.xlabel("TNNT2")
plt.ylabel("PECAM1")
plt.show()


In [ ]:
sc.pp.scale(adata_c17)
sc.tl.pca(adata_c17)
sc.pp.neighbors(adata_c17)
sc.tl.leiden(adata_c17, resolution=1, flavor="leidenalg")

sc.pl.umap(adata_c17, color=["leiden", "TNNT2", "PECAM1"])


In [ ]:
sc.tl.rank_genes_groups(adata_c17, "leiden", method="wilcoxon")
sc.pl.rank_genes_groups(adata_c17, n_genes=10)


In [ ]:
adata_c15 = cm_sub[cm_sub.obs["leiden"] == "15"].copy()


In [ ]:
sc.pp.scale(adata_c15)
sc.tl.pca(adata_c15)
sc.pp.neighbors(adata_c15)
sc.tl.leiden(adata_c15, resolution=0.5, flavor="leidenalg")

sc.pl.umap(adata_c15, color=["leiden", "TNNT2", "PECAM1", "COL1A1", "LUM"])


In [ ]:
sc.tl.rank_genes_groups(adata_c15, "leiden", method="wilcoxon")
sc.pl.rank_genes_groups(adata_c15, n_genes=20)


## Cardiomyocyte refinement and label transfer

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

# =========================================================
# CM subset
# =========================================================
cm_sub = adata[adata.obs["celltype_final"] == "CM"].copy()

sc.pp.normalize_total(cm_sub, target_sum=1e4)
sc.pp.log1p(cm_sub)

sc.tl.pca(cm_sub, n_comps=30)
sc.pp.neighbors(cm_sub, n_neighbors=15, n_pcs=30)
sc.tl.leiden(cm_sub, resolution=2.0, flavor="leidenalg")
sc.tl.umap(cm_sub)

# =========================================================
# Subclustering of ambiguous major cardiomyocyte clusters
# =========================================================
def subcluster_cm_cluster(cm_sub, cluster_id, resolution=0.5):
    ad = cm_sub[cm_sub.obs["leiden"].astype(str) == str(cluster_id)].copy()
    sc.pp.scale(ad)
    sc.tl.pca(ad)
    sc.pp.neighbors(ad)
    sc.tl.leiden(ad, resolution=resolution, flavor="leidenalg")
    sc.tl.umap(ad)
    return ad

adata_c11 = subcluster_cm_cluster(cm_sub, "11", resolution=0.5)
adata_c12 = subcluster_cm_cluster(cm_sub, "12", resolution=0.5)
adata_c15 = subcluster_cm_cluster(cm_sub, "15", resolution=0.5)
adata_c17 = subcluster_cm_cluster(cm_sub, "17", resolution=0.5)

# =========================================================
# Subclusterlabels zurück nach cm_sub
# =========================================================
cm_sub.obs["leiden"] = cm_sub.obs["leiden"].astype(str)

subcluster_objects = {
    "11": adata_c11,
    "12": adata_c12,
    "15": adata_c15,
    "17": adata_c17,
}

for main_clust, ad in subcluster_objects.items():
    ad.obs["leiden"] = ad.obs["leiden"].astype(str)
    subcol = f"leiden_{main_clust}_sub"
    cm_sub.obs[subcol] = pd.Series(index=cm_sub.obs_names, dtype="object")
    cm_sub.obs.loc[ad.obs_names, subcol] = ad.obs["leiden"].values

# =========================================================
# CM-Reannotation
# =========================================================
cm_sub.obs["celltype_reannot"] = cm_sub.obs["leiden"].astype(str)

# direkte Hauptcluster
cm_sub.obs.loc[cm_sub.obs["leiden"] == "8", "celltype_reannot"] = "FB"
cm_sub.obs.loc[cm_sub.obs["leiden"] == "16", "celltype_reannot"] = "vascular"

map_11 = {
    "0": "CM_contaminated",
    "1": "CM",
    "2": "immune",
}

map_12 = {
    "0": "vascular",
    "1": "CM_contaminated",
    "2": "CM",
    "3": "immune",
}

map_15 = {
    "0": "CM",
    "1": "vascular",
    "2": "FB",
    "3": "FB",
}

map_17 = {
    "0": "FB",
    "1": "CM",
    "2": "vascular",
    "3": "vascular",
}

cluster_maps = {
    "11": map_11,
    "12": map_12,
    "15": map_15,
    "17": map_17,
}

for main_clust, mapping in cluster_maps.items():
    subcol = f"leiden_{main_clust}_sub"
    mask = cm_sub.obs["leiden"] == main_clust

    cm_sub.obs.loc[mask, "celltype_reannot"] = (
        cm_sub.obs.loc[mask, subcol]
        .astype(str)
        .map(mapping)
        .fillna(f"{main_clust}_unassigned")
    )

# numerische Restcluster sind CM
cm_sub.obs["celltype_reannot_final"] = cm_sub.obs["celltype_reannot"].astype(str)
mask_numeric = cm_sub.obs["celltype_reannot_final"].str.fullmatch(r"\d+")
cm_sub.obs.loc[mask_numeric, "celltype_reannot_final"] = "CM"

# =========================================================
# Auf adata übertragen
# =========================================================
adata.obs["celltype_final_corrected"] = adata.obs["celltype_final"].astype(str)

common_cells = adata.obs_names.intersection(cm_sub.obs_names)

adata.obs.loc[common_cells, "celltype_final_corrected"] = (
    cm_sub.obs.loc[common_cells, "celltype_reannot_final"].astype(str)
)

adata.obs["celltype_final_corrected"] = pd.Categorical(
    adata.obs["celltype_final_corrected"],
    categories=["CM", "CM_contaminated", "FB", "vascular", "immune"]
)

print(adata.obs["celltype_final_corrected"].value_counts())


In [ ]:
# Save the cardiomyocyte-refined labels before the vascular refinement reloads this object.
adata.write("adata_global_fb_cm_cleaned_annotated.h5ad")


## Vascular compartment refinement

In [ ]:
adata = sc.read("adata_global_fb_cm_cleaned_annotated.h5ad")


## Vascular marker review and subclustering

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

def present(genes, adata):
    return [g for g in genes if g in adata.var_names]

# --- subset ---
vasc_sub = adata[adata.obs["celltype_final"] == "vascular"].copy()
vasc_sub

# --- preprocessing (Xenium-friendly) ---
sc.pp.normalize_total(vasc_sub, target_sum=1e4)
sc.pp.log1p(vasc_sub)

# --- clustering (NO HVG, NO scale) ---
sc.tl.pca(vasc_sub, n_comps=30)
sc.pp.neighbors(vasc_sub, n_neighbors=15, n_pcs=30)
sc.tl.leiden(vasc_sub, resolution=2.0, flavor="leidenalg")
sc.tl.umap(vasc_sub)

# --- marker sets ---
cm_markers = present(["TNNT2", "MYH6", "MYH7", "NPPA", "NPPB"], vasc_sub)
fb_markers = present(["COL1A1", "DCN", "LUM", "PDGFRA"], vasc_sub)
vasc_markers = present(["PECAM1", "VWF", "CD93"], vasc_sub)
imm_markers = present(["PTPRC", "CD3D", "CD3E", "NKG7"], vasc_sub)

print("CM:", cm_markers)
print("FB:", fb_markers)
print("Vascular:", vasc_markers)
print("Immune:", imm_markers)

# --- robust expression-based scoring (recommended for Xenium) ---
vasc_sub.obs["CM_mean"] = np.array(vasc_sub[:, cm_markers].X.mean(axis=1)).flatten()
vasc_sub.obs["FB_mean"] = np.array(vasc_sub[:, fb_markers].X.mean(axis=1)).flatten()
vasc_sub.obs["Vasc_mean"] = np.array(vasc_sub[:, vasc_markers].X.mean(axis=1)).flatten()
vasc_sub.obs["Immune_mean"] = np.array(vasc_sub[:, imm_markers].X.mean(axis=1)).flatten()

# --- optional scanpy scores (secondary, not primary) ---
sc.tl.score_genes(vasc_sub, cm_markers, score_name="CM_score")
sc.tl.score_genes(vasc_sub, fb_markers, score_name="FB_score")
sc.tl.score_genes(vasc_sub, vasc_markers, score_name="Vascular_score")
sc.tl.score_genes(vasc_sub, imm_markers, score_name="Immune_score")

# --- derived metrics (vascular-focused) ---
vasc_sub.obs["Vasc_minus_CM"] = vasc_sub.obs["Vasc_mean"] - vasc_sub.obs["CM_mean"]
vasc_sub.obs["Vasc_minus_FB"] = vasc_sub.obs["Vasc_mean"] - vasc_sub.obs["FB_mean"]
vasc_sub.obs["Vasc_minus_Immune"] = vasc_sub.obs["Vasc_mean"] - vasc_sub.obs["Immune_mean"]
vasc_sub.obs["FB_minus_Vasc"] = vasc_sub.obs["FB_mean"] - vasc_sub.obs["Vasc_mean"]
vasc_sub.obs["CM_minus_Vasc"] = vasc_sub.obs["CM_mean"] - vasc_sub.obs["Vasc_mean"]
vasc_sub.obs["Immune_minus_Vasc"] = vasc_sub.obs["Immune_mean"] - vasc_sub.obs["Vasc_mean"]

# --- cluster summary ---
vasc_scores = (
    vasc_sub.obs
    .groupby("leiden", observed=False)[[
        "CM_mean", "FB_mean", "Vasc_mean", "Immune_mean",
        "Vasc_minus_CM", "Vasc_minus_FB", "Vasc_minus_Immune"
    ]]
    .mean()
    .sort_values("Vasc_minus_CM", ascending=False)
)

print(vasc_scores.round(3))

# --- visualization ---
sc.pl.umap(
    vasc_sub,
    color=[
        "leiden",
        "Vasc_mean",
        "CM_mean",
        "FB_mean",
        "Immune_mean",
        "Vasc_minus_CM",
        "Vasc_minus_FB"
    ],
    vmax="p99"
)


In [ ]:
import scanpy as sc

# =========================
# Marker sets (vascular context)
# =========================
markers = {
    "Vascular_core": ["PECAM1", "VWF", "CD93", "CLDN5", "EMCN"],
    "Endothelial_activation": ["APOLD1", "RAMP2", "SOX18", "MMRN2"],
    "Perivascular_SMC": ["ACTA2", "TAGLN", "MYH11", "RGS5", "PDGFRB"],
    "CM_contamination": ["TNNT2", "MYH6", "MYH7", "NPPA"],
    "FB_contamination": ["COL1A1", "DCN", "LUM", "PDGFRA"],
    "Immune_contamination": ["PTPRC", "CD3D", "CD8A", "NKG7", "CD68", "FCGR3A"]
}

marker_list = sum(markers.values(), [])

# =========================
# 1. Marker Dotplot
# =========================
sc.pl.dotplot(
    vasc_sub,
    marker_list,
    groupby="leiden",
    standard_scale="var",
    dot_min=0.05,
    dot_max=0.8,
    cmap="Reds",
    title="Vascular subset: marker genes (endo / perivascular / contamination)"
)

# =========================
# 2. Rank genes
# =========================
sc.tl.rank_genes_groups(
    vasc_sub,
    groupby="leiden",
    method="wilcoxon",
    n_genes=10
)

# =========================
# 3. Ranked genes dotplot
# =========================
sc.pl.rank_genes_groups_dotplot(
    vasc_sub,
    n_genes=5,
    standard_scale="var",
    cmap="Reds",
    title="Vascular subset: top ranked genes per cluster"
)

# =========================
# 4. Optional combined view (markers + ranked genes)
# =========================
ranked = vasc_sub.uns["rank_genes_groups"]["names"]
top_ranked = list(set(ranked.flatten()))[:30]

combined_genes = marker_list + top_ranked

sc.pl.dotplot(
    vasc_sub,
    combined_genes,
    groupby="leiden",
    standard_scale="var",
    dot_min=0.05,
    dot_max=0.8,
    cmap="Reds",
    title="Vascular subset: markers + ranked genes"
)


In [ ]:
sc.tl.rank_genes_groups(
    vasc_sub,
    groupby="leiden",
    method="wilcoxon",
    n_genes=20
)

sc.pl.rank_genes_groups(
    vasc_sub,
    sharey=False,
    title="Ranked genes"
)


In [ ]:
import scanpy as sc
import pandas as pd

markers = {
    "Vascular_core": ["PECAM1", "VWF", "CD93"],
    "Perivascular_SMC": ["ACTA2", "TAGLN", "MYH11", "PDGFRB"],
    "CM_contamination": ["TNNT2", "MYH6", "MYH7", "NPPA"],
    "FB_contamination": ["COL1A1", "DCN", "LUM", "PDGFRA"],
    "Immune_contamination": ["PTPRC", "CD3D", "CD8A", "NKG7", "CD68", "FCGR3A"]
}

def present(genes, adata):
    return [g for g in genes if g in adata.var_names]

def analyze_cluster(adata, cluster_id, resolution=0.5, n_ranked=10):
    print(f"\n==============================")
    print(f"Cluster {cluster_id} | resolution={resolution}")
    print(f"==============================")

    # -------------------------
    # Subset
    # -------------------------
    sub = adata[adata.obs["leiden"].astype(str) == str(cluster_id)].copy()
    print(sub)

    # -------------------------
    # Preprocessing
    # -------------------------
    sc.pp.normalize_total(sub, target_sum=1e4)
    sc.pp.log1p(sub)

    sc.tl.pca(sub, n_comps=30)
    sc.pp.neighbors(sub, n_neighbors=15, n_pcs=30)
    sc.tl.leiden(sub, resolution=resolution, flavor="leidenalg")
    sc.tl.umap(sub)

    # Recompute the dendrogram for this subset
    sc.tl.dendrogram(sub, groupby="leiden")

    # -------------------------
    # Marker list
    # -------------------------
    marker_list = []
    for k in markers:
        marker_list += present(markers[k], sub)

    print("Markers used:", marker_list)

    # -------------------------
    # UMAP
    # -------------------------
    sc.pl.umap(
        sub,
        color=["leiden"],
        title=f"Cluster {cluster_id} subclustering (res={resolution})"
    )

    # -------------------------
    # Marker-Dotplot
    # -------------------------
    sc.pl.dotplot(
        sub,
        marker_list,
        groupby="leiden",
        standard_scale="var",
        dot_min=0.05,
        dot_max=0.8,
        cmap="Reds",
        title=f"Cluster {cluster_id}: vascular vs SMC vs contamination",
        dendrogram=False
    )

    # -------------------------
    # Ranked genes
    # -------------------------
    sc.tl.rank_genes_groups(
        sub,
        groupby="leiden",
        method="wilcoxon",
        n_genes=n_ranked
    )

    # Plot ranked genes
    sc.pl.rank_genes_groups_dotplot(
        sub,
        n_genes=5,
        standard_scale="var",
        cmap="Reds",
        title=f"Cluster {cluster_id}: top genes",
        dendrogram=False
    )

    # -------------------------
    # Tabelle mit ranked genes pro Subcluster
    # -------------------------
    ranked = sc.get.rank_genes_groups_df(sub, group=None)

    print("\n=== Top ranked genes per subcluster ===")
    for cl in sorted(sub.obs["leiden"].astype(str).unique(), key=lambda x: int(x)):
        top_cl = ranked[ranked["group"] == cl].head(n_ranked)
        print(f"\nSubcluster {cl}")
        print(top_cl[["names", "logfoldchanges", "pvals_adj"]])

    return sub, ranked


In [ ]:
clusters_to_check = ["8", "10", "20", "2", "14", "13", "18", "5"]

cluster_resolutions = {
    "8": 0.7,
    "10": 0.6,
    "20": 0.5,
    "2": 0.7,
    "14": 0.6,
    "13": 0.5,
    "18": 0.6,
    "5": 0.7,
}

subclusters = {}
ranked_tables = {}

vasc_sub.obs["leiden"] = vasc_sub.obs["leiden"].astype(str)

for cl in clusters_to_check:
    print(f"\n==============================")
    print(f"Cluster {cl}")
    print(f"==============================")

    res = cluster_resolutions.get(cl, 0.5)

    # analyze_cluster gibt vermutlich (sub, ranked) zurück
    sub, ranked = analyze_cluster(vasc_sub, cl, resolution=res)

    subclusters[cl] = sub
    ranked_tables[cl] = ranked

    subcol = f"leiden_{cl}_sub"

    # Spalte nur einmal anlegen, sonst überschreibst du sie in jeder Runde neu
    if subcol not in vasc_sub.obs.columns:
        vasc_sub.obs[subcol] = pd.Series(index=vasc_sub.obs_names, dtype="object")

    # Subcluster zurückschreiben
    vasc_sub.obs.loc[sub.obs_names, subcol] = sub.obs["leiden"].astype(str).values


In [ ]:
import pandas as pd

# =========================================================
# 0) Preparation
# =========================================================
vasc_sub.obs["leiden"] = vasc_sub.obs["leiden"].astype(str)

# =========================================================
# 1) Basisannotation in vasc_sub
#    Alles erstmal als vascular lassen
# =========================================================
vasc_sub.obs["celltype_reannot_vasc"] = "vascular"

# =========================================================
# 2) Finalisierte Mappings
# =========================================================
map_2 = {
    "0": "CM_contaminated",
    "1": "vascular",
    "2": "vascular",
    "3": "FB",
    "4": "CM",
}

map_5 = {
    "0": "CM",
    "1": "vascular",
    "2": "vascular",
    "3": "vascular",
}

map_8 = {
    "0": "CM",
    "1": "vascular",
    "2": "vascular",
    "3": "vascular",
}

map_10 = {
    "0": "CM_contaminated",
    "1": "vascular",
    "2": "FB",
    "3": "CM_contaminated",
}

map_13 = {
    "0": "vascular",
    "1": "vascular",
    "2": "vascular",
}

map_14 = {
    "0": "FB",
    "1": "CM_contaminated",
    "2": "CM",
    "3": "FB",
    "4": "immune",
}

map_18 = {
    "0": "vascular",
    "1": "CM_contaminated",
    "2": "CM_contaminated",
    "3": "CM_contaminated",
}

map_20 = {
    "0": "CM",
    "1": "FB",
    "2": "vascular",
}

cluster_maps = {
    "2": map_2,
    "5": map_5,
    "8": map_8,
    "10": map_10,
    "13": map_13,
    "14": map_14,
    "18": map_18,
    "20": map_20,
}

# =========================================================
# 3) Reannotation in vasc_sub anwenden
# =========================================================
for main_clust, mapping in cluster_maps.items():
    subcol = f"leiden_{main_clust}_sub"
    mask = vasc_sub.obs["leiden"] == main_clust

    if subcol not in vasc_sub.obs.columns:
        raise KeyError(f"{subcol} wurde in vasc_sub.obs nicht gefunden.")

    vasc_sub.obs.loc[mask, "celltype_reannot_vasc"] = (
        vasc_sub.obs.loc[mask, subcol]
        .astype(str)
        .map(mapping)
        .fillna(f"{main_clust}_unassigned")
    )

# =========================================================
# 4) Override: Cluster 17 komplett als CM setzen
# =========================================================
vasc_sub.obs.loc[vasc_sub.obs["leiden"] == "17", "celltype_reannot_vasc"] = "CM"

# =========================================================
# 5) Kontrolle in vasc_sub
# =========================================================
print("=== Hauptcluster vs. Reannotation in vasc_sub ===")
print(pd.crosstab(vasc_sub.obs["leiden"], vasc_sub.obs["celltype_reannot_vasc"]))

print("\n=== Finale Annotationen in vasc_sub ===")
print(vasc_sub.obs["celltype_reannot_vasc"].value_counts())

for main_clust in ["2", "5", "8", "10", "13", "14", "18", "20"]:
    subcol = f"leiden_{main_clust}_sub"
    print(f"\n=== Subcluster-Verteilung für Cluster {main_clust} ===")
    print(vasc_sub.obs.loc[vasc_sub.obs["leiden"] == main_clust, subcol].value_counts(dropna=False))

print("\n=== Cluster 17 final ===")
print(vasc_sub.obs.loc[vasc_sub.obs["leiden"] == "17", "celltype_reannot_vasc"].value_counts())

# =========================================================
# 6) Korrigierte Labels zurück nach adata übertragen
#    Basis: die schon vorhandene globale Korrektur
# =========================================================
adata.obs["celltype_final_corrected_v2"] = adata.obs["celltype_final_corrected"].astype(str)

common_cells = adata.obs_names.intersection(vasc_sub.obs_names)

adata.obs.loc[common_cells, "celltype_final_corrected_v2"] = (
    vasc_sub.obs.loc[common_cells, "celltype_reannot_vasc"].astype(str)
)

# =========================================================
# 7) Optional: als kategoriale Variable setzen
# =========================================================
adata.obs["celltype_final_corrected_v2"] = pd.Categorical(
    adata.obs["celltype_final_corrected_v2"],
    categories=["CM", "CM_contaminated", "FB", "vascular", "immune"]
)

# =========================================================
# 8) Kontrolle in adata
# =========================================================
print("\n=== Verteilung in adata nach vascular-Korrektur ===")
print(adata.obs["celltype_final_corrected_v2"].value_counts())

print("\n=== Vergleich alt vs. neu für Zellen aus vasc_sub ===")
print(pd.crosstab(
    adata.obs.loc[common_cells, "celltype_final_corrected"],
    adata.obs.loc[common_cells, "celltype_final_corrected_v2"]
))


In [ ]:
adata.obs["celltype_final_corrected"] = adata.obs["celltype_final_corrected_v2"].astype(str)


In [ ]:
adata.write("adata_celltype_corrected_with_vascular_reannotation.h5ad")


## Immune compartment refinement

In [ ]:
adata = sc.read("adata_celltype_corrected_with_vascular_reannotation.h5ad")


### Immune marker review

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

def present(genes, adata):
    return [g for g in genes if g in adata.var_names]

# --- subset ---
immune_sub = adata[adata.obs["celltype_final_corrected"] == "immune"].copy()
immune_sub

# --- preprocessing (Xenium-friendly) ---
sc.pp.normalize_total(immune_sub, target_sum=1e4)
sc.pp.log1p(immune_sub)

# --- clustering (NO HVG, NO scale) ---
sc.tl.pca(immune_sub, n_comps=30)
sc.pp.neighbors(immune_sub, n_neighbors=15, n_pcs=30)
sc.tl.leiden(immune_sub, resolution=1.6, flavor="leidenalg")
sc.tl.umap(immune_sub)

# --- marker sets ---
cm_markers = present(["TNNT2", "MYH6", "MYH7", "NPPA", "NPPB"], immune_sub)
fb_markers = present(["COL1A1", "DCN", "LUM", "PDGFRA"], immune_sub)
vasc_markers = present(["PECAM1", "VWF", "CD93"], immune_sub)
imm_markers = present(["PTPRC", "CD3D", "CD3E", "NKG7"], immune_sub)

print("CM:", cm_markers)
print("FB:", fb_markers)
print("Vascular:", vasc_markers)
print("Immune:", imm_markers)

# --- robust expression-based scoring ---
immune_sub.obs["CM_mean"] = np.array(immune_sub[:, cm_markers].X.mean(axis=1)).flatten()
immune_sub.obs["FB_mean"] = np.array(immune_sub[:, fb_markers].X.mean(axis=1)).flatten()
immune_sub.obs["Vasc_mean"] = np.array(immune_sub[:, vasc_markers].X.mean(axis=1)).flatten()
immune_sub.obs["Immune_mean"] = np.array(immune_sub[:, imm_markers].X.mean(axis=1)).flatten()

# --- optional scanpy scores ---
sc.tl.score_genes(immune_sub, cm_markers, score_name="CM_score")
sc.tl.score_genes(immune_sub, fb_markers, score_name="FB_score")
sc.tl.score_genes(immune_sub, vasc_markers, score_name="Vascular_score")
sc.tl.score_genes(immune_sub, imm_markers, score_name="Immune_score")

# --- derived metrics (immune-focused) ---
immune_sub.obs["Immune_minus_CM"] = immune_sub.obs["Immune_mean"] - immune_sub.obs["CM_mean"]
immune_sub.obs["Immune_minus_FB"] = immune_sub.obs["Immune_mean"] - immune_sub.obs["FB_mean"]
immune_sub.obs["Immune_minus_Vasc"] = immune_sub.obs["Immune_mean"] - immune_sub.obs["Vasc_mean"]

immune_sub.obs["CM_minus_Immune"] = immune_sub.obs["CM_mean"] - immune_sub.obs["Immune_mean"]
immune_sub.obs["FB_minus_Immune"] = immune_sub.obs["FB_mean"] - immune_sub.obs["Immune_mean"]
immune_sub.obs["Vasc_minus_Immune"] = immune_sub.obs["Vasc_mean"] - immune_sub.obs["Immune_mean"]

# --- cluster summary ---
immune_scores = (
    immune_sub.obs
    .groupby("leiden", observed=False)[[
        "CM_mean", "FB_mean", "Vasc_mean", "Immune_mean",
        "Immune_minus_CM", "Immune_minus_FB", "Immune_minus_Vasc"
    ]]
    .mean()
    .sort_values("Immune_minus_CM", ascending=False)
)

print(immune_scores.round(3))

# --- visualization ---
sc.pl.umap(
    immune_sub,
    color=[
        "leiden",
        "Immune_mean",
        "CM_mean",
        "FB_mean",
        "Vasc_mean",
        "Immune_minus_CM",
        "Immune_minus_Vasc"
    ],
    vmax="p99"
)


In [ ]:
import scanpy as sc

# =========================
# Marker sets (immune context)
# =========================
markers = {
    "Immune_core": ["PTPRC", "CD3D", "CD3E", "NKG7", "CD68", "FCGR3A"],
    "T_NK": ["CD3D", "CD3E", "CD8A", "NKG7"],
    "Myeloid": ["CD68", "FCGR3A", "MPEG1", "LST1", "CD163"],
    "CM_contamination": ["TNNT2", "MYH6", "MYH7", "NPPA"],
    "FB_contamination": ["COL1A1", "DCN", "LUM", "PDGFRA"],
    "Vascular_contamination": ["PECAM1", "VWF", "CD93"]
}

marker_list = []
for genes in markers.values():
    marker_list += present(genes, immune_sub)

# =========================
# 1. Marker Dotplot
# =========================
sc.pl.dotplot(
    immune_sub,
    marker_list,
    groupby="leiden",
    standard_scale="var",
    dot_min=0.05,
    dot_max=0.8,
    cmap="Reds",
    title="Immune subset: marker genes (immune / contamination)"
)

# =========================
# 2. Rank genes
# =========================
sc.tl.rank_genes_groups(
    immune_sub,
    groupby="leiden",
    method="wilcoxon",
    n_genes=10
)

# =========================
# 3. Ranked genes dotplot
# =========================
sc.pl.rank_genes_groups_dotplot(
    immune_sub,
    n_genes=5,
    standard_scale="var",
    cmap="Reds",
    title="Immune subset: top ranked genes per cluster",
    dendrogram=False
)

# =========================
# 4. Optional combined view (markers + ranked genes)
# =========================
ranked = immune_sub.uns["rank_genes_groups"]["names"]
top_ranked = list(set(ranked.flatten()))[:30]

combined_genes = marker_list + top_ranked

sc.pl.dotplot(
    immune_sub,
    combined_genes,
    groupby="leiden",
    standard_scale="var",
    dot_min=0.05,
    dot_max=0.8,
    cmap="Reds",
    title="Immune subset: markers + ranked genes",
    dendrogram=False
)


In [ ]:
sc.tl.rank_genes_groups(
    immune_sub,
    groupby="leiden",
    method="wilcoxon",
    n_genes=20
)

sc.pl.rank_genes_groups(
    immune_sub,
    sharey=False,
    title="Ranked genes"
)


In [ ]:
import scanpy as sc
import pandas as pd

# =========================================================
# Marker sets for immune-cell subclustering
# =========================================================
markers = {
    "Immune_core": ["PTPRC", "CD3D", "CD3E", "CD8A", "NKG7", "CD68", "FCGR3A"],
    "T_NK": ["CD3D", "CD3E", "CD8A", "NKG7"],
    "Myeloid": ["CD68", "FCGR3A", "MPEG1", "LST1", "CD163"],
    "CM_contamination": ["TNNT2", "MYH6", "MYH7", "NPPA"],
    "FB_contamination": ["COL1A1", "DCN", "LUM", "PDGFRA"],
    "Vascular_contamination": ["PECAM1", "VWF", "CD93"]
}

def present(genes, adata):
    return [g for g in genes if g in adata.var_names]

def analyze_cluster(adata, cluster_id, resolution=0.5, n_ranked=10):
    print(f"\n==============================")
    print(f"Cluster {cluster_id} | resolution={resolution}")
    print(f"==============================")

    # -------------------------
    # Subset
    # -------------------------
    sub = adata[adata.obs["leiden"].astype(str) == str(cluster_id)].copy()
    print(sub)

    # -------------------------
    # Preprocessing
    # -------------------------
    sc.pp.normalize_total(sub, target_sum=1e4)
    sc.pp.log1p(sub)

    sc.tl.pca(sub, n_comps=30)
    sc.pp.neighbors(sub, n_neighbors=15, n_pcs=30)
    sc.tl.leiden(sub, resolution=resolution, flavor="leidenalg")
    sc.tl.umap(sub)

    # -------------------------
    # Marker list
    # -------------------------
    marker_list = []
    for k in markers:
        marker_list += present(markers[k], sub)

    print("Markers used:", marker_list)

    print("\nSubcluster sizes:")
    print(sub.obs["leiden"].value_counts())

    # -------------------------
    # UMAP
    # -------------------------
    sc.pl.umap(
        sub,
        color=["leiden"],
        title=f"Cluster {cluster_id} subclustering (res={resolution})"
    )

    # -------------------------
    # Marker-Dotplot
    # -------------------------
    sc.pl.dotplot(
        sub,
        marker_list,
        groupby="leiden",
        standard_scale="var",
        dot_min=0.05,
        dot_max=0.8,
        cmap="Reds",
        title=f"Cluster {cluster_id}: immune vs contamination",
        dendrogram=False
    )

    # -------------------------
    # Ranked genes nur wenn >1 Subcluster
    # -------------------------
    n_subclusters = sub.obs["leiden"].nunique()

    if n_subclusters >= 2:
        sc.tl.rank_genes_groups(
            sub,
            groupby="leiden",
            method="wilcoxon",
            n_genes=n_ranked
        )

        sc.pl.rank_genes_groups_dotplot(
            sub,
            n_genes=5,
            standard_scale="var",
            cmap="Reds",
            title=f"Cluster {cluster_id}: top genes",
            dendrogram=False
        )

        ranked = sc.get.rank_genes_groups_df(sub, group=None)

        print("\n=== Top ranked genes per subcluster ===")
        for scl in sorted(sub.obs["leiden"].astype(str).unique(), key=lambda x: int(x)):
            top_cl = ranked[ranked["group"] == scl].head(n_ranked)
            print(f"\nSubcluster {scl}")
            print(top_cl[["names", "logfoldchanges", "pvals_adj"]])

    else:
        ranked = None
        print("\nNur ein Subcluster gefunden -> keine ranked genes zwischen Subclustern möglich.")

    return sub, ranked


In [ ]:
# =========================================================
# Clusters selected for review within the immune compartment
# =========================================================
clusters_to_check = ["1", "5", "8", "10", "11", "15", "17"]

cluster_resolutions = {
    "1": 0.5,
    "5": 0.5,
    "8": 0.5,
    "10": 0.5,
    "11": 0.5,
    "15": 0.5,
    "17": 0.5,
}

subclusters = {}
ranked_tables = {}

immune_sub.obs["leiden"] = immune_sub.obs["leiden"].astype(str)

for cl in clusters_to_check:
    print(f"\n==============================")
    print(f"Cluster {cl}")
    print(f"==============================")

    res = cluster_resolutions.get(cl, 0.5)

    sub, ranked = analyze_cluster(immune_sub, cl, resolution=res)

    subclusters[cl] = sub
    ranked_tables[cl] = ranked

    subcol = f"leiden_{cl}_sub"

    if subcol not in immune_sub.obs.columns:
        immune_sub.obs[subcol] = pd.Series(index=immune_sub.obs_names, dtype="object")

    immune_sub.obs.loc[sub.obs_names, subcol] = sub.obs["leiden"].astype(str).values


### Apply immune subcluster labels and create v3 annotation

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

# =========================================================
# Immune subset
# =========================================================
immune_sub = adata[adata.obs["celltype_final_corrected_v2"] == "immune"].copy()

sc.pp.normalize_total(immune_sub, target_sum=1e4)
sc.pp.log1p(immune_sub)

sc.tl.pca(immune_sub, n_comps=30)
sc.pp.neighbors(immune_sub, n_neighbors=15, n_pcs=30)
sc.tl.leiden(immune_sub, resolution=1.6, flavor="leidenalg")
sc.tl.umap(immune_sub)

# =========================================================
# Subclustering helper
# =========================================================
markers = {
    "Immune_core": ["PTPRC", "CD3D", "CD3E", "CD8A", "NKG7", "CD68", "FCGR3A"],
    "T_NK": ["CD3D", "CD3E", "CD8A", "NKG7"],
    "Myeloid": ["CD68", "FCGR3A", "MPEG1", "LST1", "CD163"],
    "CM_contamination": ["TNNT2", "MYH6", "MYH7", "NPPA"],
    "FB_contamination": ["COL1A1", "DCN", "LUM", "PDGFRA"],
    "Vascular_contamination": ["PECAM1", "VWF", "CD93"],
}

def present(genes, adata):
    return [g for g in genes if g in adata.var_names]

def analyze_cluster(adata, cluster_id, resolution=0.5, n_ranked=10):
    sub = adata[adata.obs["leiden"].astype(str) == str(cluster_id)].copy()

    sc.pp.normalize_total(sub, target_sum=1e4)
    sc.pp.log1p(sub)

    sc.tl.pca(sub, n_comps=30)
    sc.pp.neighbors(sub, n_neighbors=15, n_pcs=30)
    sc.tl.leiden(sub, resolution=resolution, flavor="leidenalg")
    sc.tl.umap(sub)

    marker_list = []
    for k in markers:
        marker_list += present(markers[k], sub)

    sc.pl.umap(sub, color=["leiden"], title=f"Cluster {cluster_id} subclustering")

    sc.pl.dotplot(
        sub,
        marker_list,
        groupby="leiden",
        standard_scale="var",
        dot_min=0.05,
        dot_max=0.8,
        cmap="Reds",
        title=f"Cluster {cluster_id}: immune / contamination",
        dendrogram=False,
    )

    n_subclusters = sub.obs["leiden"].nunique()

    if n_subclusters >= 2:
        sc.tl.rank_genes_groups(sub, groupby="leiden", method="wilcoxon", n_genes=n_ranked)

        sc.pl.rank_genes_groups_dotplot(
            sub,
            n_genes=5,
            standard_scale="var",
            cmap="Reds",
            title=f"Cluster {cluster_id}: top genes",
            dendrogram=False,
        )

        ranked = sc.get.rank_genes_groups_df(sub, group=None)

        print("\n=== Top ranked genes per subcluster ===")
        for scl in sorted(sub.obs["leiden"].astype(str).unique(), key=lambda x: int(x)):
            top_cl = ranked[ranked["group"] == scl].head(n_ranked)
            print(f"\nSubcluster {scl}")
            print(top_cl[["names", "logfoldchanges", "pvals_adj"]])
    else:
        ranked = None
        print("Nur ein Subcluster gefunden.")

    return sub, ranked

# =========================================================
# Immune clusters subclustern
# =========================================================
clusters_to_check = ["1", "5", "8", "10", "11", "15", "17"]

cluster_resolutions = {
    "1": 0.5,
    "5": 0.6,
    "8": 0.6,
    "10": 0.6,
    "11": 0.5,
    "15": 0.6,
    "17": 0.6,
}

subclusters = {}
ranked_tables = {}

immune_sub.obs["leiden"] = immune_sub.obs["leiden"].astype(str)

for cl in clusters_to_check:
    res = cluster_resolutions.get(cl, 0.5)

    sub, ranked = analyze_cluster(immune_sub, cl, resolution=res)

    subclusters[cl] = sub
    ranked_tables[cl] = ranked

    subcol = f"leiden_{cl}_sub"

    if subcol not in immune_sub.obs.columns:
        immune_sub.obs[subcol] = pd.Series(index=immune_sub.obs_names, dtype="object")

    immune_sub.obs.loc[sub.obs_names, subcol] = sub.obs["leiden"].astype(str).values

# =========================================================
# Immune-Reannotation
# =========================================================
immune_sub.obs["celltype_immune_refined"] = "immune"

immune_maps = {
    "1": {
        "0": "immune_contaminated",
        "1": "FB",
        "2": "immune_contaminated",
    },
    "5": {
        "0": "immune_contaminated",
        "1": "immune_contaminated",
        "2": "immune",
        "3": "immune",
    },
    "8": {
        "0": "immune",
        "1": "immune_contaminated",
        "2": "CM",
    },
    "10": {
        "0": "immune",
        "1": "CM",
        "2": "immune_contaminated",
    },
    "11": {
        "0": "immune",
        "1": "immune",
        "2": "immune",
        "3": "immune_contaminated",
    },
    "15": {
        "0": "immune",
        "1": "vascular",
        "2": "immune",
        "3": "immune_contaminated",
    },
    "17": {
        "0": "CM",
        "1": "immune_contaminated",
        "2": "immune_contaminated",
    },
}

for main_clust, mapping in immune_maps.items():
    subcol = f"leiden_{main_clust}_sub"
    mask = immune_sub.obs["leiden"] == main_clust

    immune_sub.obs.loc[mask, "celltype_immune_refined"] = (
        immune_sub.obs.loc[mask, subcol]
        .astype(str)
        .map(mapping)
        .fillna(f"{main_clust}_unassigned")
    )

# direkte Hauptcluster
direct_map = {
    "4": "FB",
    "6": "CM",
    "9": "CM",
}

for cl, label in direct_map.items():
    immune_sub.obs.loc[immune_sub.obs["leiden"] == cl, "celltype_immune_refined"] = label

# =========================================================
# Auf adata übertragen
# =========================================================
adata.obs["celltype_final_corrected_v3"] = adata.obs["celltype_final_corrected_v2"].astype(str)

common_cells = adata.obs_names.intersection(immune_sub.obs_names)

adata.obs.loc[common_cells, "celltype_final_corrected_v3"] = (
    immune_sub.obs.loc[common_cells, "celltype_immune_refined"].astype(str)
)

adata.obs["celltype_final_corrected_v3"] = pd.Categorical(
    adata.obs["celltype_final_corrected_v3"],
    categories=[
        "CM",
        "CM_contaminated",
        "FB",
        "vascular",
        "immune",
        "immune_contaminated",
    ],
)

print(adata.obs["celltype_final_corrected_v3"].value_counts())


## Save the v3 checkpoint

In [ ]:
# Final cell type annotation

print("Final cell type counts:")
print(adata.obs["celltype_final_corrected_v3"].value_counts())

# Save annotated AnnData object
adata.write_h5ad("adata_celltype_corrected_v3.h5ad")

print("Saved adata_celltype_corrected_v3.h5ad")


In [ ]:
from pathlib import Path
import scanpy as sc

checkpoint = Path("adata_celltype_corrected_v3.h5ad")
if not checkpoint.exists():
    raise FileNotFoundError(f"Expected checkpoint was not created: {checkpoint.resolve()}")
saved = sc.read_h5ad(checkpoint, backed="r")
required = "celltype_final_corrected_v3"
if required not in saved.obs.columns:
    raise KeyError(f"Required annotation column {required!r} is absent from {checkpoint.name}")
print(f"Validated {checkpoint.name}: {saved.n_obs:,} cells; annotation column {required!r} present.")
saved.file.close()
